# EV fleet exploration

Driving, home connection, battery energy, and fleet demand in five-minute intervals.

Install the project with `python -m pip install -e ".[notebook]"` from the repository root before opening this notebook. Run the cells in order. The modeling decisions, equations, and limitations are in [the README](../README.md).

In [ ]:
from IPython.display import display

from ev_fleet_sim import SimulationConfig, load_archetypes, run_scenario
from ev_fleet_sim.plotting import make_agent_figure, make_population_figure

display(
    load_archetypes()[
        [
            "archetype",
            "population_share",
            "miles_per_year",
            "battery_capacity_kwh",
            "plugin_frequency_per_day",
            "charging_policy",
        ]
    ]
)

## A reproducible example

The notebook runs a small fleet automatically. Increase the vehicle count or number of days below. This is a new run starting at target SoC, with no warm-up.

In [ ]:
result = run_scenario(SimulationConfig(number_of_agents=24, days=3, seed=42))
display(result.calibration.round(3))
display(result.checks)
make_population_figure(result.summary).show()

## Sessions and one vehicle

A connection can include multiple charging sessions and periods with no power draw. Filter by session type before summing session energy. A left-censored connection was already present at the start, so exclude it when asking for observed plug-in events.

In [ ]:
plug_ins = result.sessions.query("session_type == 'connection' and not left_censored")
display(
    plug_ins[["agent_id", "archetype", "start", "end", "soc_start", "grid_energy_kwh"]].head(10)
)
make_agent_figure(result.data, result.agents.agent_id.iloc[0]).show()

## Interactive scenarios

Choose archetypes from a fixed sampled population and edit driving probabilities. Click **Run scenario** to compute. Lower driving probability increases mean active-day distance to preserve expected annual mileage, so some settings may be infeasible. Errors clear old outputs. Widgets require a live Jupyter kernel.

Population mix can be changed through the Python API shown in the README. The controls below filter an existing fleet and do not resample it.

In [ ]:
from ev_fleet_sim.explorer import build_explorer

explorer = build_explorer(number_of_agents=100, seed=42)
display(explorer.widget)

## Inspect or export the latest interactive run

The interval table records SoC at both boundaries, energy in kWh, and interval-average grid power in kW. Uncomment the save call to export the current result.

In [ ]:
if explorer.result is None:
    print("Click Run scenario above to generate an interactive result.")
else:
    display(explorer.result.data.head())
    # explorer.result.save("../outputs/interactive", charts=True)

## Interpretation

This is a stochastic behavioral model with an energy-balance approximation. SoC bands describe variation across vehicles. They are not confidence intervals. Mileage matches source inputs in expectation, and observed connection rates can differ from daily opportunity probabilities. The initial connection, full knowledge of the next trip, and the finite run boundary affect short scenarios.

The current model has no public charging, V2G dispatch, temperature model, charging taper, degradation, or geographic grid model. Stored energy is not a measure of exportable energy.